# Exploring ERA5 Reanalysis Temperature Data over CONUS

**Course:** Climate and Climate Change  
**Topic:** Working with gridded reanalysis data and spatial visualization

---

## Learning Objectives

By the end of this exercise, you will be able to:
1. Understand what **ERA5 reanalysis** data is and why it's used in climate science
2. Open and inspect a **NetCDF** file using `xarray`
3. Extract and slice spatial data by region and time
4. Create a publication-quality **filled contour map** of surface temperature
5. Interpret spatial patterns in temperature across the continental United States (CONUS)

---

## Background

### What is ERA5?

[ERA5](https://www.ecmwf.int/en/forecasts/dataset/ecmwf-reanalysis-v5) is the fifth generation **global atmospheric reanalysis** produced by the European Centre for Medium-Range Weather Forecasts (ECMWF). Reanalysis data combines historical observations (weather stations, radiosondes, satellites) with a numerical weather model to produce a consistent, gridded record of the atmosphere dating back to 1940.

ERA5 provides:
- Hourly data at ~31 km horizontal resolution (~0.25°)
- Hundreds of atmospheric and surface variables
- Global coverage from 1940 to near-present

It is one of the most widely used datasets in climate research.

### What is NetCDF?

**NetCDF** (Network Common Data Form) is the standard file format for storing multidimensional scientific data (e.g., temperature varying in latitude, longitude, pressure level, and time). Nearly all climate and atmospheric datasets use this format.

---

## Step 1: Set Up the Environment

We first install and import the libraries we need. If you are running this on **Google Colab**, the cell below will install any missing packages automatically.

In [ ]:
# Install required packages (only needed on Colab or fresh environments)
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

required = ["cartopy"]
for pkg in required:
    install(pkg)

print("All packages ready!")

In [ ]:
# Core scientific libraries
import numpy as np
import xarray as xr

# Plotting
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Cartopy for map projections and geographic features
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Suppress minor warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully.")

---

## Step 2: Download the Dataset from Google Drive

The ERA5 dataset for this exercise is hosted on Google Drive. We use the `gdown` library to download it directly into our Colab session.

In [ ]:
import gdown
import os

# -------------------------------------------------------
# The variable below holds the file ID for the google file
# which the gdown package will download from Google Drive
GDRIVE_FILE_ID = "1XtK7DYuIPBKlhCte3gtHC900gFEWzUEk"
# -------------------------------------------------------

OUTPUT_FILENAME = "era5_conus_t2m.nc"

if not os.path.exists(OUTPUT_FILENAME):
    url = f"https://drive.google.com/uc?id={GDRIVE_FILE_ID}"
    print(f"Downloading dataset from Google Drive...")
    gdown.download(url, OUTPUT_FILENAME, quiet=False)
    print(f"Download complete: {OUTPUT_FILENAME}")
else:
    print(f"File already exists: {OUTPUT_FILENAME}")

---

## Step 3: Open and Inspect the Dataset

We use `xarray` to open the NetCDF file. `xarray` is the standard Python library for working with labeled, multidimensional arrays — think of it as pandas for gridded scientific data.

In [ ]:
# Open the NetCDF file with xarray
ds = xr.open_dataset(OUTPUT_FILENAME)

# Print a summary of the dataset
print(ds)

### 🔍 Understanding the Dataset Structure

Take a moment to read the output above. Identify:
- **Dimensions**: What axes does the data have? (e.g., `time`, `latitude`, `longitude`)
- **Coordinates**: What are the actual values along each dimension?
- **Data variables**: What variables are stored? (We expect `t2m` = 2-meter air temperature)
- **Attributes**: Metadata like units, source, and creation date

> **❓ Question 1:** What are the latitude and longitude bounds of this dataset? Does it cover all of CONUS?

In [ ]:
# Inspect the 2-meter temperature variable specifically
t2m = ds['t2m']  # ERA5 variable name for 2-meter air temperature
print(t2m)
print(f"\nUnits: {t2m.attrs.get('units', 'not specified')}")
print(f"Long name: {t2m.attrs.get('long_name', 'not specified')}")
print(f"Shape: {t2m.shape}")
print(f"Dimensions: {t2m.dims}")

In [ ]:
# Check the time dimension
print("Available time steps:")
print(ds.valid_time.values)

---

## Step 4: Prepare the Data for Plotting

ERA5 stores temperature in **Kelvin**. We'll convert to **Celsius** for more intuitive interpretation. We'll also select a single time step to plot.

In [ ]:
# Select the first time step (modify the index to explore other times)
time_idx = 0
selected_time = ds.valid_time.values[time_idx]
print(f"Plotting data for: {selected_time}")

# Select that time step and convert Kelvin → Celsius
t2m_slice = ds['t2m'].isel(valid_time=time_idx) - 273.15
t2m_slice.attrs['units'] = '°C'

# Pull out coordinate arrays for convenience
lons = ds['longitude'].values
lats = ds['latitude'].values

print(f"Temperature range: {float(t2m_slice.min()):.1f} to {float(t2m_slice.max()):.1f} °C")

> **❓ Question 2:** What is the temperature range across CONUS for your selected time step? Does the range make physical sense given the season?

---

## Step 5: Plot the Spatial Temperature Field

Now we create a filled contour map using `matplotlib` and `cartopy`. Cartopy handles the map projection and geographic features (coastlines, state borders, etc.).

In [ ]:
# --- Plot Configuration ---
# Map projection: LambertConformal is standard for CONUS display
proj = ccrs.LambertConformal(
    central_longitude=-96.0,
    central_latitude=37.5,
    standard_parallels=(33, 45)
)

fig, ax = plt.subplots(
    figsize=(12, 7),
    subplot_kw={'projection': proj}
)

# Set map extent to CONUS [west, east, south, north]
ax.set_extent([-125, -66, 23, 50], crs=ccrs.PlateCarree())

# --- Filled Contour Plot ---
# Define contour levels (every 2°C from -30 to 40°C)
levels = np.arange(-30, 42, 2)

cf = ax.contourf(
    lons, lats, t2m_slice.values,
    levels=levels,
    cmap='RdBu_r',          # Diverging colormap: blue=cold, red=warm
    transform=ccrs.PlateCarree(),  # Data is in lat/lon, not projected
    extend='both'           # Extend colorbar for out-of-range values
)

# --- Contour Lines (optional overlay) ---
cl = ax.contour(
    lons, lats, t2m_slice.values,
    levels=levels[::5],     # Only every 5th level for readability
    colors='k',
    linewidths=0.5,
    transform=ccrs.PlateCarree()
)
ax.clabel(cl, inline=True, fontsize=7, fmt='%d°C')

# --- Geographic Features ---
ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.8, linestyle='-')
ax.add_feature(cfeature.STATES, linewidth=0.4, edgecolor='gray')
ax.add_feature(cfeature.OCEAN, color='lightcyan', zorder=0)
ax.add_feature(cfeature.LAKES, color='lightcyan', zorder=0)

# --- Gridlines ---
gl = ax.gridlines(
    crs=ccrs.PlateCarree(),
    draw_labels=True,
    linewidth=0.5,
    color='gray',
    alpha=0.5,
    linestyle='--'
)
gl.top_labels = False
gl.right_labels = False
gl.xlocator = mticker.FixedLocator([-120, -110, -100, -90, -80, -70])
gl.ylocator = mticker.FixedLocator([25, 30, 35, 40, 45, 50])

# --- Colorbar ---
cbar = plt.colorbar(cf, ax=ax, orientation='horizontal', pad=0.04, shrink=0.8)
cbar.set_label('2-meter Air Temperature (°C)', fontsize=11)
cbar.ax.tick_params(labelsize=9)

# --- Title ---
time_str = str(selected_time)[:16]  # Trim to YYYY-MM-DDTHH:MM
ax.set_title(
    f'ERA5 2-Meter Air Temperature over CONUS\n{time_str} UTC',
    fontsize=13, fontweight='bold', pad=10
)

plt.tight_layout()
plt.savefig('era5_t2m_conus.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved as era5_t2m_conus.png")

---

## Step 6: Explore the Data Further

### 6a. Plot a Different Time Step

Modify `time_idx` in Step 4 and re-run Steps 4 and 5 to visualize temperature at a different time.

> **❓ Question 3:** How does the temperature pattern change between your two selected times? What meteorological processes might explain these differences?

### 6b. Compute and Plot the Time-Mean Temperature

Instead of a single snapshot, let's compute the **temporal mean** across all time steps.

In [ ]:
# Compute time mean
t2m_mean = ds['t2m'].mean(dim='time') - 273.15
t2m_mean.attrs['units'] = '°C'

print(f"Time-mean temperature range: {float(t2m_mean.min()):.1f} to {float(t2m_mean.max()):.1f} °C")

# --- Plot ---
fig, ax = plt.subplots(figsize=(12, 7), subplot_kw={'projection': proj})
ax.set_extent([-125, -66, 23, 50], crs=ccrs.PlateCarree())

cf = ax.contourf(
    lons, lats, t2m_mean.values,
    levels=np.arange(-30, 42, 2),
    cmap='RdBu_r',
    transform=ccrs.PlateCarree(),
    extend='both'
)

ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
ax.add_feature(cfeature.BORDERS, linewidth=0.8)
ax.add_feature(cfeature.STATES, linewidth=0.4, edgecolor='gray')
ax.add_feature(cfeature.OCEAN, color='lightcyan', zorder=0)

gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                  linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
gl.top_labels = False
gl.right_labels = False

cbar = plt.colorbar(cf, ax=ax, orientation='horizontal', pad=0.04, shrink=0.8)
cbar.set_label('2-meter Air Temperature (°C)', fontsize=11)

n_times = len(ds.time)
ax.set_title(
    f'ERA5 Time-Mean 2-Meter Temperature over CONUS\n(mean over {n_times} time steps)',
    fontsize=13, fontweight='bold'
)

plt.tight_layout()
plt.savefig('era5_t2m_mean_conus.png', dpi=150, bbox_inches='tight')
plt.show()

### 6c. First vs. Last Time Step — Side-by-Side Comparison

One of the most powerful ways to communicate climate change is a direct **before/after** comparison. Here we plot the earliest and most recent time steps in the dataset side-by-side using a **shared, locked colorbar** so the two panels are visually comparable.

> **❓ Question 4:** Compare the two panels carefully. Where do you see the largest temperature differences? Are the changes spatially uniform, or do some regions warm (or cool) more than others? What might explain any regional contrasts you observe?

In [ ]:
# --- Select first and last time steps ---
t2m_first = ds['t2m'].isel(valid_time=0)  - 273.15
t2m_last  = ds['t2m'].isel(valid_time=-1) - 273.15

time_first = str(ds.valid_time.values[0])[:16]
time_last  = str(ds.valid_time.values[-1])[:16]

print(f"First time step : {time_first} UTC")
print(f"Last  time step : {time_last}  UTC")

# --- Shared colorbar limits: use the combined data range ---
vmin = float(min(t2m_first.min(), t2m_last.min()))
vmax = float(max(t2m_first.max(), t2m_last.max()))
# Round outward to nearest 2 °C for clean contour levels
vmin = np.floor(vmin / 2) * 2
vmax = np.ceil(vmax  / 2) * 2
levels = np.arange(vmin, vmax + 2, 2)

print(f"\nShared color range: {vmin:.0f} to {vmax:.0f} °C")

# --- Map projection (same as previous steps) ---
proj = ccrs.LambertConformal(
    central_longitude=-96.0,
    central_latitude=37.5,
    standard_parallels=(33, 45)
)

# --- Figure: 1 row, 2 columns ---
fig, axes = plt.subplots(
    nrows=1, ncols=2,
    figsize=(20, 6),
    subplot_kw={'projection': proj},
    constrained_layout=True
)

panels = [
    (axes[0], t2m_first, time_first, 'earliest'),
    (axes[1], t2m_last,  time_last,  'most recent'),
]

for ax, data, time_str, label in panels:
    ax.set_extent([-125, -66, 23, 50], crs=ccrs.PlateCarree())

    # Filled contours — same levels and colormap on both panels
    cf = ax.contourf(
        lons, lats, data.values,
        levels=levels,
        cmap='RdBu_r',
        transform=ccrs.PlateCarree(),
        extend='both'
    )

    # Contour lines every 10 °C
    cl = ax.contour(
        lons, lats, data.values,
        levels=levels[::5],
        colors='k', linewidths=0.5,
        transform=ccrs.PlateCarree()
    )
    ax.clabel(cl, inline=True, fontsize=7, fmt='%d°C')

    # Geographic features
    ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
    ax.add_feature(cfeature.BORDERS,   linewidth=0.8)
    ax.add_feature(cfeature.STATES,    linewidth=0.4, edgecolor='gray')
    ax.add_feature(cfeature.OCEAN,     color='lightcyan', zorder=0)
    ax.add_feature(cfeature.LAKES,     color='lightcyan', zorder=0)

    # Gridlines (labels only on left panel to avoid clutter)
    gl = ax.gridlines(
        crs=ccrs.PlateCarree(), draw_labels=(ax is axes[0]),
        linewidth=0.5, color='gray', alpha=0.5, linestyle='--'
    )
    gl.top_labels   = False
    gl.right_labels = False
    gl.xlocator = mticker.FixedLocator([-120, -110, -100, -90, -80, -70])
    gl.ylocator = mticker.FixedLocator([25, 30, 35, 40, 45, 50])

    ax.set_title(
        f'ERA5 2-m Temperature ({label})\n{time_str} UTC',
        fontsize=12, fontweight='bold'
    )

# Shared horizontal colorbar spanning both panels
cbar = fig.colorbar(
    cf, ax=axes, orientation='horizontal',
    pad=0.04, shrink=0.7, aspect=40
)
cbar.set_label('2-meter Air Temperature (°C)', fontsize=11)
cbar.ax.tick_params(labelsize=9)

fig.suptitle(
    'ERA5 2-Meter Temperature over CONUS: Earliest vs. Most Recent Time Step',
    fontsize=14, fontweight='bold', y=1.02
)

plt.savefig('era5_t2m_first_vs_last.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved as era5_t2m_first_vs_last.png")

---

## Step 7: Zonal Mean Temperature Profile

A **zonal mean** averages over all longitudes at each latitude, revealing the meridional temperature gradient — one of the fundamental drivers of atmospheric circulation.

In [ ]:
# Zonal (longitude) mean of the time-mean temperature
zonal_mean = t2m_mean.mean(dim='longitude')

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(zonal_mean.values, lats, color='firebrick', linewidth=2)
ax.axvline(0, color='k', linewidth=0.8, linestyle='--', label='0°C')

ax.set_xlabel('Temperature (°C)', fontsize=12)
ax.set_ylabel('Latitude (°N)', fontsize=12)
ax.set_title('Zonal Mean 2-m Temperature vs. Latitude (CONUS domain)', fontsize=11)
ax.grid(True, alpha=0.4)
ax.legend()

plt.tight_layout()
plt.savefig('era5_zonal_mean.png', dpi=150, bbox_inches='tight')
plt.show()

> **❓ Question 4:** Describe the relationship between latitude and temperature shown in the zonal mean plot. What is the approximate temperature gradient (°C per degree of latitude) across CONUS? How does this relate to the driving of mid-latitude weather systems?

---

## Step 8: Temperature Trend Analysis

The maps in previous steps show the **mean state** of temperature, but climate science is fundamentally about how that state is **changing over time**. Here we compute a linear trend at every grid point across all time steps, which reveals *where* warming (or cooling) is occurring and at what rate.

### What is a linear trend?

At each grid point $(lat, lon)$, we fit a simple linear regression:

$$T(t) = a \cdot t + b$$

where $t$ is time (in years), $b$ is the intercept, and $a$ is the **trend** in °C per year. A positive value of $a$ means warming; negative means cooling.

We use `numpy.polyfit` applied across the time dimension with `xarray.apply_ufunc` so the operation vectorizes efficiently over the full spatial grid without a slow Python loop.

> **Note:** A meaningful trend requires sufficient temporal span. If your dataset covers only a few years or a single season, the computed trends will reflect short-term variability more than a true climate signal. Keep this in mind when interpreting the results.

In [ ]:
# -----------------------------------------------------------------------
# Step 8a: Compute the linear temperature trend at every grid point
# -----------------------------------------------------------------------

# Convert the time coordinate to a numeric axis (decimal years)
# This makes the trend slope interpretable as °C per year
import pandas as pd

times_dt = pd.to_datetime(ds.valid_time.values)
# Decimal year: year + (day-of-year / days-in-year)
t_years = np.array(
    [t.year + (t.day_of_year - 1) / (366 if t.is_leap_year else 365)
     for t in times_dt]
)
n_years = t_years[-1] - t_years[0]

print(f"Dataset spans {n_years:.2f} years")
print(f"  from {times_dt[0].strftime('%Y-%m-%d')}")
print(f"  to   {times_dt[-1].strftime('%Y-%m-%d')}")
print(f"  ({len(t_years)} time steps)")

# Convert temperature to °C for the whole dataset
t2m_C = ds['t2m'] - 273.15

# Define a function that fits a 1st-degree polynomial along the last axis
# and returns only the slope (index 0 of the coefficient array)
def linear_trend(y, t):
    """Return the linear trend slope (°C/year) for a 1-D time series y."""
    return np.polyfit(t, y, 1)[0]

# apply_ufunc vectorizes linear_trend over all (lat, lon) grid points
trend_map = xr.apply_ufunc(
    linear_trend,
    t2m_C,
    kwargs={'t': t_years},
    input_core_dims=[['time']],   # operate along the time axis
    vectorize=True,               # loop over all other dims automatically
    dask='parallelized',
    output_dtypes=[float]
)

# Also scale to °C per decade for more intuitive presentation
trend_map_dec = trend_map * 10.0

print(f"\nTrend range across CONUS:")
print(f"  {float(trend_map_dec.min()):.3f} to {float(trend_map_dec.max()):.3f} °C/decade")

### 8b: Spatial Map of the Warming Trend

The map below shows the local linear trend (°C per decade) at every ERA5 grid point. A **diverging colormap centered on zero** makes it easy to distinguish warming from cooling regions at a glance.

In [ ]:
# -----------------------------------------------------------------------
# Step 8b: Spatial map of temperature trend (°C / decade)
# -----------------------------------------------------------------------

proj = ccrs.LambertConformal(
    central_longitude=-96.0,
    central_latitude=37.5,
    standard_parallels=(33, 45)
)

fig, ax = plt.subplots(figsize=(12, 7), subplot_kw={'projection': proj})
ax.set_extent([-125, -66, 23, 50], crs=ccrs.PlateCarree())

# Symmetric colorbar limits: round up to nearest 0.5 °C/decade
abs_max = float(np.ceil(max(abs(float(trend_map_dec.min())),
                             abs(float(trend_map_dec.max()))) * 2) / 2)
trend_levels = np.linspace(-abs_max, abs_max, 41)

cf = ax.contourf(
    lons, lats, trend_map_dec.values,
    levels=trend_levels,
    cmap='RdBu_r',                    # red = warming, blue = cooling
    transform=ccrs.PlateCarree(),
    extend='both'
)

# Zero-line contour to clearly mark the boundary between warming and cooling
ax.contour(
    lons, lats, trend_map_dec.values,
    levels=[0],
    colors='k', linewidths=1.2,
    transform=ccrs.PlateCarree()
)

# Geographic features
ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
ax.add_feature(cfeature.BORDERS,   linewidth=0.8)
ax.add_feature(cfeature.STATES,    linewidth=0.4, edgecolor='gray')
ax.add_feature(cfeature.OCEAN,     color='lightcyan', zorder=0)
ax.add_feature(cfeature.LAKES,     color='lightcyan', zorder=0)

gl = ax.gridlines(
    crs=ccrs.PlateCarree(), draw_labels=True,
    linewidth=0.5, color='gray', alpha=0.5, linestyle='--'
)
gl.top_labels   = False
gl.right_labels = False
gl.xlocator = mticker.FixedLocator([-120, -110, -100, -90, -80, -70])
gl.ylocator = mticker.FixedLocator([25, 30, 35, 40, 45, 50])

cbar = plt.colorbar(cf, ax=ax, orientation='horizontal', pad=0.04, shrink=0.8)
cbar.set_label('Temperature Trend (°C / decade)', fontsize=11)
cbar.ax.tick_params(labelsize=9)
# Mark zero on the colorbar
cbar.ax.axvline(0, color='k', linewidth=1.0)

date_range = f"{times_dt[0].strftime('%Y-%m-%d')} – {times_dt[-1].strftime('%Y-%m-%d')}"
ax.set_title(
    f'ERA5 2-Meter Temperature Trend over CONUS\n'
    f'Linear trend (°C / decade)  |  {date_range}',
    fontsize=13, fontweight='bold'
)

plt.tight_layout()
plt.savefig('era5_t2m_trend_spatial.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved as era5_t2m_trend_spatial.png")

> **❓ Question 5:** Identify two or three regions over CONUS that show the strongest warming trend. Can you associate those regions with known geographic or land-surface features (e.g., the Arctic-adjacent northern tier, the arid Southwest, urban areas)? Are there any regions showing a cooling or near-zero trend, and what might explain them?

### 8c: Zonal Mean Trend Profile

Just as we computed the zonal mean temperature in Step 7, we can compute the **zonal mean of the trend** — averaging the per-grid-point trend across all longitudes at each latitude. This collapses the spatial map into a single curve showing how the warming rate varies with latitude.

In [ ]:
# -----------------------------------------------------------------------
# Step 8c: Zonal mean of the temperature trend vs. latitude
# -----------------------------------------------------------------------

zonal_trend = trend_map_dec.mean(dim='longitude')   # average over all longitudes

# Also compute ± 1 standard deviation across longitudes to show spread
zonal_trend_std = trend_map_dec.std(dim='longitude')

fig, ax = plt.subplots(figsize=(7, 6))

# Shaded uncertainty band (±1σ across longitudes)
ax.fill_betweenx(
    lats,
    (zonal_trend - zonal_trend_std).values,
    (zonal_trend + zonal_trend_std).values,
    color='salmon', alpha=0.35, label='±1σ across longitudes'
)

# Zonal mean trend line
ax.plot(
    zonal_trend.values, lats,
    color='firebrick', linewidth=2.5, label='Zonal mean trend'
)

# Zero-trend reference line
ax.axvline(0, color='k', linewidth=1.0, linestyle='--', label='No trend (0 °C/decade)')

# Domain-mean trend as a single number for annotation
domain_mean_trend = float(trend_map_dec.mean())
ax.axvline(
    domain_mean_trend, color='darkorange', linewidth=1.2, linestyle=':',
    label=f'CONUS mean: {domain_mean_trend:+.3f} °C/decade'
)

ax.set_xlabel('Temperature Trend (°C / decade)', fontsize=12)
ax.set_ylabel('Latitude (°N)', fontsize=12)
ax.set_ylim(lats.min(), lats.max())
ax.set_title(
    f'Zonal Mean Temperature Trend vs. Latitude (CONUS domain)\n'
    f'{times_dt[0].strftime("%Y-%m-%d")} – {times_dt[-1].strftime("%Y-%m-%d")}',
    fontsize=11
)
ax.grid(True, alpha=0.4)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('era5_t2m_trend_zonal.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"CONUS domain-mean trend: {domain_mean_trend:+.3f} °C/decade")

> **❓ Question 6:** Does the warming trend appear to be uniform with latitude, or is it stronger at higher or lower latitudes within CONUS? This phenomenon — where high latitudes warm faster than the tropics — is known as **Arctic amplification**. Do you see any evidence of it here? What physical mechanisms might drive amplified warming at higher latitudes?

---

## Summary

In this exercise you:
- Downloaded a real **ERA5 reanalysis** NetCDF dataset
- Inspected its structure using `xarray`
- Converted units and selected spatial/temporal slices
- Created a **filled contour map** with state/country borders using `cartopy`
- Computed time means and a zonal mean temperature profile
- Computed a **pixel-wise linear temperature trend** across all time steps
- Visualized the trend spatially and as a **zonal mean profile** with uncertainty bounds

These are core skills used daily in climate research. The same workflow applies to any gridded atmospheric variable (precipitation, wind, humidity, etc.) from any reanalysis or climate model output.

---

## 📝 Assignment Questions

Answer the following in a short write-up (1–2 paragraphs each):

1. Describe the large-scale temperature pattern you observe over CONUS. What physical factors (latitude, topography, proximity to ocean) seem to control the spatial distribution?

2. How does the snapshot (single time step) compare to the time-mean map? What does this tell us about **weather variability** vs. **climatological mean state**?

3. ERA5 is a **reanalysis**, not direct observations. What are the advantages and potential limitations of using reanalysis data in climate studies?

4. Compare the first and last time step maps from Step 6c. Are the differences you observe consistent with the direction of warming expected from climate change, or could they reflect natural variability (e.g., seasonal cycle, weather patterns)? How would you design a better comparison to isolate a long-term climate signal?

5. The spatial trend map in Step 8b shows where warming is fastest. Identify two specific regions with notably high or low trend values and propose a physical explanation for each.

6. The zonal mean trend profile (Step 8c) includes a ±1σ shaded band. What does this spread represent physically? If the shaded band crosses zero at a given latitude, what does that imply about confidence in the trend at that latitude?

7. *(Bonus)* Modify the notebook to plot a different ERA5 variable (e.g., `sp` for surface pressure, or `u10`/`v10` for 10-m winds if available in the dataset). Describe what you find.

---

## Additional Resources

- [ERA5 documentation (ECMWF)](https://confluence.ecmwf.int/display/CKB/ERA5)
- [xarray documentation](https://docs.xarray.dev/)
- [Cartopy documentation](https://scitools.org.uk/cartopy/docs/latest/)
- [Copernicus Climate Data Store](https://cds.climate.copernicus.eu/) — where you can download ERA5 data yourself

In [ ]:
# Clean up: close the dataset
ds.close()
print("Dataset closed. Notebook complete!")